In [1]:
from pathlib import Path
import pandas as pd
import mne
from tqdm import tqdm

ROOT = Path("/media/dll-1/SSD 4TB/EEG Datasets/nmt_4k_split")
EDF_DIRS = ["train", "eval"]

EXPECTED_21 = [
    "FP1","FP2","F7","F3","FZ","F4","F8","T3","C3","CZ","C4","T4","T5","P3","PZ","P4","T6","O1","O2","A1","A2","ECG"
]

def norm(ch): 
    return str(ch).strip().upper().replace(" ", "")

rows = []
for split in EDF_DIRS:
    p = ROOT / split
    edfs = list(p.rglob("*.edf"))
    for edf in tqdm(edfs, desc=f"Reading {split}"):
        try:
            raw = mne.io.read_raw_edf(str(edf), preload=False, verbose=False)

            chs = [norm(c) for c in raw.ch_names]
            sfreq = float(raw.info["sfreq"])
            dur_sec = float(raw.times[-1]) if len(raw.times) else float("nan")

            missing = sorted(set(EXPECTED_21) - set(chs))
            extra = sorted(set(chs) - set(EXPECTED_21))

            rows.append({
                "split": split,
                "file_id": edf.stem,
                "path": str(edf),
                "sfreq": sfreq,
                "duration_min": dur_sec/60.0,
                "n_channels": len(chs),
                "missing_expected": len(missing),
                "extra_channels": len(extra),
                "missing_list": " ".join(missing)[:300],  # truncate
                "extra_list": " ".join(extra)[:300],
            })
        except Exception as e:
            rows.append({
                "split": split,
                "file_id": edf.stem,
                "path": str(edf),
                "error": str(e),
            })

df = pd.DataFrame(rows)
out = Path("./nmt4k_step1_out")
out.mkdir(parents=True, exist_ok=True)
df.to_csv(out / "edf_integrity_summary.csv", index=False)

print("Saved:", out / "edf_integrity_summary.csv")
print("\nQuick counts:")
print(df.groupby("split")["file_id"].count())
print("\nSampling rate distribution:")
print(df.groupby("sfreq")["file_id"].count().sort_index())
print("\nWorst missing_expected (top 10):")

if "missing_expected" in df.columns:
    print(df.sort_values("missing_expected", ascending=False)[
        ["split","file_id","sfreq","duration_min","n_channels","missing_expected","extra_channels"]
    ].head(10))


Reading eval: 100%|██████████| 1000/1000 [00:01<00:00, 769.51it/s]

Saved: nmt4k_step1_out/edf_integrity_summary.csv

Quick counts:
split
eval     1000
train    3000
Name: file_id, dtype: int64

Sampling rate distribution:
sfreq
200.0    4000
Name: file_id, dtype: int64

Worst missing_expected (top 10):
      split          file_id  sfreq  duration_min  n_channels  \
0     train  mh_2019_0001221  200.0     10.199917          22   
2671  train  mh_2019_0002296  200.0     13.266583          22   
2658  train  mh_2021_0000158  200.0     29.399917          22   
2659  train  mh_2022_0000306  200.0     21.466583          22   
2660  train  mh_2020_0000456  200.0     19.966583          22   
2661  train  mh_2020_0000185  200.0     14.999917          22   
2662  train  mh_2024_0004217  200.0     22.166583          22   
2663  train  mh_2019_0000890  200.0     12.083250          22   
2664  train  mh_2021_0000182  200.0     25.399917          22   
2665  train  mh_2024_0003659  200.0     17.783250          22   

      missing_expected  extra_channels  
0     

In [2]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import mne
from tqdm import tqdm
from collections import Counter

# =========================
# PATHS (edit if needed)
# =========================
ROOT = Path("/media/dll-1/SSD 4TB/EEG Datasets/nmt_4k_split")
EDF_DIRS = ["train", "eval"]
CSV_DIR = ROOT / "csv"     # change if your annotations folder has a different name
# CSV_DIR = ROOT / "csv_fixed_channels_fczpz"
CSV_DIR = ROOT / "csv_fixed_channels_fczpz_updated"

OUT = Path("./nmt4k_validation_out")
OUT.mkdir(parents=True, exist_ok=True)

# =========================
# CSV COLUMNS (edit if needed)
# =========================
COL_FILE_START = "File Start"
COL_START_TIME = "Start time"
COL_END_TIME   = "End time"
COL_CHANS      = "Channel names"
COL_LABEL      = "Comment"

# =========================
# EXPECTED CHANNELS (your 22)
# =========================
EXPECTED_CHANNELS = {
    "FP1","FP2","F7","F3","FZ","F4","F8","T3","C3","CZ","C4","T4","T5",
    "P3","PZ","P4","T6","O1","O2","A1","A2","ECG"
}

def norm(ch): 
    return str(ch).strip().upper().replace(" ", "")

# =========================
# TIME PARSING
# =========================
def parse_time_hhmmss_ms(t: str):
    """Accept HH:MM:SS:ms or HH:MM:SS.ms or HH:MM:SS -> seconds since midnight."""
    if t is None or (isinstance(t, float) and np.isnan(t)):
        return None
    s = str(t).strip()
    if not s:
        return None
    m = re.match(r"^(\d{1,2}):(\d{1,2}):(\d{1,2})(?:[:\.](\d{1,3}))?$", s)
    if not m:
        return None
    hh, mm, ss, ms = m.groups()
    hh = int(hh); mm = int(mm); ss = int(ss)
    ms_i = int(ms) if ms is not None else 0
    if ms is not None and len(ms) == 1: ms_i *= 100
    if ms is not None and len(ms) == 2: ms_i *= 10
    return hh*3600 + mm*60 + ss + (ms_i/1000.0)

def seconds_since_start(file_start, event_time):
    fs = parse_time_hhmmss_ms(file_start)
    et = parse_time_hhmmss_ms(event_time)
    if fs is None or et is None:
        return None
    # midnight rollover (rare but safe)
    if et < fs:
        et += 24*3600
    return float(et - fs)

# =========================
# LABEL MERGING DICTIONARY
# =========================
ab_label_dict_detail = [
    ['Normal', 
     'Beta Wave','Beta waves', 'beta waves', 'Beta Wave', 'beta activity',
     'Alpha Coma', 'alpha coma'],

    ['Sharp Wave',
     'sharp waves', 'sharp wave', 'generalized sharp waves', 'generalized sharp waves discharge', 'Sharp Wave',
     'sharp and wave', 'sharp wave discharges', 'sharp wave pre del both',
     'Triphasic Wave', 'triphasic waves', 'periodic & semiperiodic & triphasic wave',
     'periodic & semiperiodic & triphasic wave  & complexes',
     'periodic & semiperiodic & triphasic wave & sharp & slow wave & complexes',
     'semi periodic triphasic', 'semi periodic triphasic wave', 'semiperiodic triphasic wave complexes',
     'triphaic waves', 'triphasic waves ( sharp waves)', 'triphasic waves (sharp waves)',
     'semiperiodic triphasic wavecomplexes', 'semiperiodic  triphasic wave',"triphasic waves'",
     'Burst Suppression', 'burst suppression', 'burst suppression pattern',
     'Sharp and slow Wave', 'focal sharp and slow wave', 'sharp and delta waves', 'sharp & delta wave',
     'sharp & slow wave activity', 'sharp and delta slow wave', 'sharp and slow wave activity',
     'sharp and slow wave discharges', 'slow and sharp wave', 'sharp & delta slow waves',
     'sharp and deta wave', 'sharp ans slow wave discharges', 'sharp and delta slow waves',
     'sharp and delta waves', 'sharp and delta wave', 'sharp and slow waves', 'sharp and slow wave'
    ],

    ['Delta Slow Wave',
     'delta slow wave', 'delta waves', 'delta slow waves', 'slowing wave',
     'generalized paroxymal delta slow waves', 'generalized paroxysmal delta slow waves',
     'generalized parosysmal delta slow waves', 'generalized delta slow waves', 'slow waves',
     'paroxysmal delta slow waves', 'paroxysmal generalized delta slow waves',
     'Delta', 'delta wave', 'changegeneralized delta slow waves', 'dela slow waves',
     'delta slow waves with muscle artifacts', 'delta slowing', 'frontal delta slowing',
     'frontal slow wave', 'frontal slow wave activity', 'generalized delta slow waves discharge',
     'generalized delta slowing', 'generalized slowing', 'generalized slowing in delta range',
     'mild  slowing', 'mild generalized slowing', 'slow wave', 'slow wave activity', 'slow wave discharges',
     'delta slow waves\\', 'delta wavest', 'delta low waves', 'delta slo waves', 'delta sloww aves',
     'delta sloww waves', 'delta slowwaves', 'generaliaed slowing', 'generalize slowing',
     'generalized  slowing', 'generalizied slowing', 'generalizwd delta slowing',
     'generilized slowing', 'generlized slowing', 'generralized slowing',
     'mild generilized slowing', 'mild generlized slowing',
     'delta prev','Generalized slowing mixed with Artifacts'
    ],

    ['Spike and Wave',
     '2 hertz slow spike and wave discharge', 'spike wave', 'spikes',
     'generalized paroxysmal spike and wave discharge', 'generalized paroxymal spike and wave discharge',
     'fragmented spike and wave discharge', 'generalized paroxysmal 3 hertz spike and wave discharge',
     'generalized spike and wave discharge', 'generalized spike and wave discharges',
     'spike and wave', 'spike and wave discharge', 'spike and waves', 'generalized spike and wave',
     'spike and wave ', ' spike and wave discharge', 'spike', 'spiek and wave',
     'Spike and Wave Discharge', '3hz spike and wave', '3hz spike and wave discharges',
     'brust of spike and wave', 'spike & wave discharges', 'spike and wave activity',
     'spike and wave discharges', 'spike snd wave', 'slow spike and wave',
     'spike and wave dischages', 'Rolandic Spikes', 'rolandic spike', 'rolandic spikes'
    ],

    ['Polyspike',
     'generalized plolyspikes discharge', 'generalized polyspike pattern',
     'polyspike and wave discharges', 'polyspike wave discharges', 'polyspikes & wave',
     'polypspikes and wave', 'generalized polyspikes discharge',
     'generalized polyspike discharge', 'polyspikes discharge', 'polyspikes and wave',
     'polyspikes', 'polyspike and wave'
    ],

    ['Spike Wave and Polyspike Wave',
     'generalized spike & polyspike pattern', 'generalized spike & polyspike wave',
     'spike & poilyspike wave', 'spike & polyspike', 'spike & polyspike and wave',
     'spike & polyspike wave', 'spike &polyspike and wave discharges',
     'spike / polyspike and wave discharges', 'spike/polyspike and wave discharges'
    ],

    ['Theta Wave',
     'theta waves', 'Theta Slow Wave', 'generalized slowing in theta range',
     'mild generilized slowing in theta range', 'theta slow waves', 'theta wavs'
    ],

    ['Delta and Theta Wave',
     'delta and theta waves', 'delta-theta slowing', 'frontal dela to theta slowing',
     'generalized slowing in delta to theta range', 'generalized slowing in delta-thta range'
    ],

    ['Spike And Delta Slow Waves',
     'spike and delta slow waves', 'spike and wave discharges followed by generalized slowing',
     'spike and wave discharges followed by slowing',
     'spike and wave discharges with generalized slowing'
    ],

    ['Low Voltage',
     'low voltage', 'no waveform', 'low  voltage', 'low voltage suppression',
     'no electrical activity', 'no voltage', 'low voltagev', 'loww voltage'
    ],

    ['Artifacts',
     'artifact', 'blink artifacts', 'frontal area artifacts', 'Artifracts', 'articats', 'artifacds'
    ],

    ['Unknown', 'No Comment', 'delete previous', 'nan', 'Unknown']
]

def normalize_label(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip().lower()
    s = s.replace("\\", " ").replace("'", " ").replace('"', " ")
    s = re.sub(r"[_/]+", " ", s)
    s = re.sub(r"[^a-z0-9&. ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def build_variant_to_canonical(groups):
    m = {}
    for g in groups:
        canonical = g[0].strip()
        for v in g:
            k = normalize_label(v)
            if k:
                m[k] = canonical
    return m

variant2canon = build_variant_to_canonical(ab_label_dict_detail)

def map_to_canonical(label: str, default="Unknown"):
    k = normalize_label(label)
    return variant2canon.get(k, default)

print("Setup complete.")


Setup complete.


In [3]:
edf_lookup = {}   # file_id -> (split, path, duration_sec, channel_set)
bad_edfs = []

for split in EDF_DIRS:
    edf_files = list((ROOT / split).rglob("*.edf"))
    for edf in tqdm(edf_files, desc=f"Indexing EDFs ({split})"):
        file_id = edf.stem
        try:
            raw = mne.io.read_raw_edf(str(edf), preload=False, verbose=False)
            dur_sec = float(raw.times[-1]) if len(raw.times) else float("nan")
            chset = set(norm(c) for c in raw.ch_names)
            edf_lookup[file_id] = (split, edf, dur_sec, chset)
        except Exception as e:
            bad_edfs.append((split, file_id, str(edf), str(e)))

print("Readable EDFs:", len(edf_lookup))
print("Unreadable EDFs:", len(bad_edfs))

pd.DataFrame(bad_edfs, columns=["split","file_id","path","error"]).to_csv(OUT/"unreadable_edfs.csv", index=False)
print("Saved:", OUT/"unreadable_edfs.csv")


Indexing EDFs (eval): 100%|██████████| 1000/1000 [00:01<00:00, 788.91it/s]

Readable EDFs: 4000
Unreadable EDFs: 0
Saved: nmt4k_validation_out/unreadable_edfs.csv


In [4]:
csv_paths = list(CSV_DIR.rglob("*.csv"))
csv_ids = {p.stem for p in csv_paths}
edf_ids = set(edf_lookup.keys())

# mapping overview
mapping = pd.DataFrame({"file_id": sorted(edf_ids | csv_ids)})
mapping["has_readable_edf"] = mapping["file_id"].isin(edf_ids)
mapping["has_csv"] = mapping["file_id"].isin(csv_ids)
mapping.to_csv(OUT/"edf_csv_mapping.csv", index=False)
print("Saved:", OUT/"edf_csv_mapping.csv")
print("\nMapping counts:")
print(mapping[["has_readable_edf","has_csv"]].value_counts())

def load_and_prepare_csv(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]

    # forward fill header-level metadata
    if COL_FILE_START in df.columns:
        df[COL_FILE_START] = df[COL_FILE_START].ffill()

    # compute rel times
    df["start_sec"] = df.apply(lambda r: seconds_since_start(r[COL_FILE_START], r[COL_START_TIME]), axis=1)
    df["end_sec"]   = df.apply(lambda r: seconds_since_start(r[COL_FILE_START], r[COL_END_TIME]), axis=1)
    df["duration_sec"] = df["end_sec"] - df["start_sec"]

    # channels list
    def parse_ch_list(x):
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return []
        items = [i for i in str(x).replace(",", " ").split() if i.strip()]
        return [norm(i) for i in items]

    df["channels"] = df[COL_CHANS].apply(parse_ch_list)

    # raw label + merged label
    df["label_raw"] = df[COL_LABEL].astype(str).str.strip()
    df["label_merged"] = df["label_raw"].apply(map_to_canonical)

    return df

file_summary_rows = []
event_tables = []

for csv_path in tqdm(csv_paths, desc="Validating CSVs"):
    file_id = csv_path.stem

    if file_id not in edf_lookup:
        file_summary_rows.append({
            "file_id": file_id,
            "status": "csv_without_readable_edf",
            "csv_path": str(csv_path),
        })
        continue

    split, edf_path, dur_sec, chset = edf_lookup[file_id]

    try:
        df = load_and_prepare_csv(csv_path)

        # time validity
        df["valid_time"] = (
            df["start_sec"].notna() & df["end_sec"].notna() &
            (df["start_sec"] >= 0) &
            (df["end_sec"] <= dur_sec + 1e-3) &
            (df["duration_sec"] > 0)
        )

        # channel validity checks
        df["invalid_channels"] = df["channels"].apply(lambda lst: [c for c in lst if c not in EXPECTED_CHANNELS])
        df["missing_in_edf"]   = df["channels"].apply(lambda lst: [c for c in lst if c not in chset])

        n_events = len(df)
        n_valid = int(df["valid_time"].sum())
        density_per_min = float(n_valid / max(dur_sec/60.0, 1e-9))

        file_summary_rows.append({
            "file_id": file_id,
            "split": split,
            "edf_path": str(edf_path),
            "csv_path": str(csv_path),
            "edf_duration_min": dur_sec/60.0,
            "n_events": n_events,
            "n_valid_events": n_valid,
            "event_density_per_min": density_per_min,
            "median_event_duration_sec": float(np.nanmedian(df.loc[df["valid_time"], "duration_sec"])) if n_valid else np.nan,
            "p25_event_duration_sec": float(np.nanpercentile(df.loc[df["valid_time"], "duration_sec"], 25)) if n_valid else np.nan,
            "p75_event_duration_sec": float(np.nanpercentile(df.loc[df["valid_time"], "duration_sec"], 75)) if n_valid else np.nan,
            "n_rows_invalid_time": int((~df["valid_time"]).sum()),
            "n_rows_invalid_channels": int((df["invalid_channels"].apply(len) > 0).sum()),
            "n_rows_channels_missing_in_edf": int((df["missing_in_edf"].apply(len) > 0).sum()),
        })

        # store compact event table
        df_out = df[
            [COL_FILE_START, COL_START_TIME, COL_END_TIME,
             "start_sec","end_sec","duration_sec",
             "label_raw","label_merged",
             "channels","valid_time","invalid_channels","missing_in_edf"]
        ].copy()
        df_out["file_id"] = file_id
        df_out["split"] = split
        event_tables.append(df_out)

    except Exception as e:
        file_summary_rows.append({
            "file_id": file_id,
            "split": split,
            "edf_path": str(edf_path),
            "csv_path": str(csv_path),
            "status": "error",
            "error": str(e),
        })

files_df = pd.DataFrame(file_summary_rows)
files_df.to_csv(OUT/"annotation_file_summary.csv", index=False)
print("Saved:", OUT/"annotation_file_summary.csv")

# Create merged event table
if event_tables:
    ev = pd.concat(event_tables, ignore_index=True)
    ev.to_csv(OUT/"annotation_event_table_merged.csv", index=False)
    print("Saved:", OUT/"annotation_event_table_merged.csv")
else:
    ev = None
    print("No event tables created (no CSVs matched readable EDFs).")

files_df.head()


Saved: nmt4k_validation_out/edf_csv_mapping.csv

Mapping counts:
has_readable_edf  has_csv
True              False      3043
                  True        957
dtype: int64


Validating CSVs: 100%|██████████| 957/957 [00:06<00:00, 158.89it/s]


Saved: nmt4k_validation_out/annotation_file_summary.csv
Saved: nmt4k_validation_out/annotation_event_table_merged.csv


,file_id,split,edf_path,csv_path,edf_duration_min,n_events,n_valid_events,event_density_per_min,median_event_duration_sec,p25_event_duration_sec,p75_event_duration_sec,n_rows_invalid_time,n_rows_invalid_channels,n_rows_channels_missing_in_edf
0,mh_2024_0003619,eval,/media/dll-1/SSD 4TB/EEG Datasets/nmt_4k_split...,/media/dll-1/SSD 4TB/EEG Datasets/nmt_4k_split...,15.016583,67,67,4.461734,6.945,4.9090,12.2365,0,0,0
1,mh_2021_0000219,eval,/media/dll-1/SSD 4TB/EEG Datasets/nmt_4k_split...,/media/dll-1/SSD 4TB/EEG Datasets/nmt_4k_split...,30.199917,105,105,3.476831,13.408,9.7700,14.4690,0,0,0
2,mh_2021_0000764,eval,/media/dll-1/SSD 4TB/EEG Datasets/nmt_4k_split...,/media/dll-1/SSD 4TB/EEG Datasets/nmt_4k_split...,19.999917,103,103,5.150021,1.200,1.0085,1.5085,0,0,0
3,mh_2019_0000406,train,/media/dll-1/SSD 4TB/EEG Datasets/nmt_4k_split...,/media/dll-1/SSD 4TB/EEG Datasets/nmt_4k_split...,13.816583,182,167,12.086925,1.364,0.8440,2.7920,15,0,0
4,mh_2019_0001450,train,/media/dll-1/SSD 4TB/EEG Datasets/nmt_4k_split...,/media/dll-1/SSD 4TB/EEG Datasets/nmt_4k_split...,14.499917,11,9,0.620693,0.494,0.4810,0.5300,2,0,0


In [5]:
if ev is None:
    print("No events to summarize.")
else:
    # merged label distribution (valid events only)
    merged_label_counts = ev.loc[ev["valid_time"], "label_merged"].value_counts()
    merged_label_counts.to_csv(OUT/"label_distribution_merged.csv", header=["count"])
    print("Saved:", OUT/"label_distribution_merged.csv")

    # channel counts (valid only)
    c = Counter()
    for chans in ev.loc[ev["valid_time"], "channels"]:
        for ch in chans:
            if ch in EXPECTED_CHANNELS:
                c[ch] += 1

    ch_df = pd.DataFrame(sorted(c.items(), key=lambda x: x[1], reverse=True),
                         columns=["channel","count"])
    ch_df.to_csv(OUT/"channel_event_counts.csv", index=False)
    print("Saved:", OUT/"channel_event_counts.csv")

    print("\nTop merged labels:")
    display(merged_label_counts.head(20))

    print("\nTop channels:")
    display(ch_df.head(25))

    # Useful quick health checks
    print("\nFile-level summary:")
    display(files_df[["n_events","n_valid_events","n_rows_invalid_time","n_rows_invalid_channels"]].describe())


Saved: nmt4k_validation_out/label_distribution_merged.csv
Saved: nmt4k_validation_out/channel_event_counts.csv

Top merged labels:


Delta Slow Wave                  18458
Spike and Wave                   14643
Sharp Wave                       13084
Low Voltage                       6997
Theta Wave                        6026
Unknown                           2995
Artifacts                          233
Normal                             187
Delta and Theta Wave               162
Spike And Delta Slow Waves          92
Polyspike                           72
Spike Wave and Polyspike Wave        7
Name: label_merged, dtype: int64


Top channels:


,channel,count
0,C4,46791
1,F4,46688
2,P4,46559
3,F8,46537
4,FP2,46201
5,F3,46079
6,O2,46078
7,F7,45924
8,O1,45830
9,C3,45779



File-level summary:


,n_events,n_valid_events,n_rows_invalid_time,n_rows_invalid_channels
count,957.000000,957.000000,957.000000,957.0
mean,75.901776,65.784744,10.117032,0.0
std,95.948517,93.087598,25.587877,0.0
min,1.000000,0.000000,0.000000,0.0
25%,16.000000,10.000000,0.000000,0.0
50%,49.000000,35.000000,0.000000,0.0
75%,101.000000,89.000000,6.000000,0.0
max,1182.000000,1182.000000,244.000000,0.0


In [6]:
import numpy as np
import pandas as pd

assert ev is not None

bad = ev[~ev["valid_time"]].copy()

# Build a single "reason" column (priority order)
def reason_row(r):
    if pd.isna(r["start_sec"]) or pd.isna(r["end_sec"]):
        return "parse_failed_or_missing_time"
    if (r["end_sec"] <= r["start_sec"]):
        return "end_before_or_equal_start"
    if (r["start_sec"] < 0) or (r["end_sec"] < 0):
        return "negative_time"
    if ("edf_duration_sec" in r.index) and (pd.notna(r["edf_duration_sec"])) and (r["end_sec"] > r["edf_duration_sec"]):
        return "end_exceeds_edf"
    return "other"

bad["invalid_reason"] = bad.apply(reason_row, axis=1)

display(bad["invalid_reason"].value_counts())

# Save for appendix
bad.to_csv(OUT/"invalid_time_rows_with_reasons.csv", index=False)
print("Saved:", OUT/"invalid_time_rows_with_reasons.csv")


other    9682
Name: invalid_reason, dtype: int64

Saved: nmt4k_validation_out/invalid_time_rows_with_reasons.csv


In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

# --- Configuration ---
CSV_DIR = Path("/media/dll-1/SSD 4TB/EEG Datasets/nmt_4k_split/csv_fixed_channels_fczpz_updated")
# Ensure we have the event table 'ev' from your previous validation steps
if "ev" not in globals() or ev is None:
    raise ValueError("Please run the validation code that generates the 'ev' table first.")

# --- Logic to Categorize the "Other" Issues ---
def classify_issue(row):
    # 1. Check Time Logic
    if pd.isna(row.get("start_sec")) or pd.isna(row.get("end_sec")):
        return "Still Parsing Error (Should be 0)"
    
    if row["end_sec"] <= row["start_sec"]:
        return "Negative Duration (End <= Start)"
    
    if "edf_duration_sec" in row and pd.notna(row["edf_duration_sec"]):
        if row["end_sec"] > row["edf_duration_sec"]:
            return "Event Exceeds EDF Duration"

    # 2. Check Channel Logic
    # (Assuming 'invalid_channels' column exists from your validation notebook)
    if "invalid_channels" in row and len(row["invalid_channels"]) > 0:
        return f"Invalid Channel Names: {row['invalid_channels']}"
        
    if "missing_in_edf" in row and len(row["missing_in_edf"]) > 0:
        return f"Channel Missing in EDF: {row['missing_in_edf']}"

    return "OK"

# Apply classification to all rows
print("Classifying remaining issues...")
ev["detailed_issue"] = ev.apply(classify_issue, axis=1)

# Filter for rows that are NOT "OK"
problems = ev[ev["detailed_issue"] != "OK"].copy()

# --- Summary Report ---
print(f"\nTotal problematic rows found: {len(problems)}")

if not problems.empty:
    print("\n--- Top Issue Categories ---")
    print(problems["detailed_issue"].value_counts().head(10))
    
    print("\n--- Example Files for Each Issue ---")
    for issue in problems["detailed_issue"].unique():
        sample_file = problems[problems["detailed_issue"] == issue].iloc[0]["file_id"]
        print(f"Issue: '{issue}' -> Check file: {sample_file}")

Classifying remaining issues...

Total problematic rows found: 0


In [8]:
# Filter rows with the reason "other"
bad_other = bad[bad["invalid_reason"] == "other"].copy()

# --- Display rows with "other" invalid reason ---
print(f"Number of rows with 'other' invalid reason: {len(bad_other)}")

# Show the rows with potential issues
display(bad_other[["file_id", "start_sec", "end_sec", "invalid_reason", "File Start", "Start time", "End time"]])

# --- Check for missing or malformed times ---
# Check which columns have missing or malformed times
missing_columns = bad_other[["file_id", "Start time", "End time", "File Start"]].isna().sum()
print("Missing columns count:")
print(missing_columns)

# Optionally, you can print it too
print(f"Rows with 'other' issue:\n", bad_other.head(20))


Number of rows with 'other' invalid reason: 9682


,file_id,start_sec,end_sec,invalid_reason,File Start,Start time,End time
442,mh_2019_0000406,834.474,835.080,other,10:46:06,11:00:00:474,11:00:01:080
443,mh_2019_0000406,835.404,835.902,other,10:46:06,11:00:01:404,11:00:01:902
444,mh_2019_0000406,836.941,837.818,other,10:46:06,11:00:02:941,11:00:03:818
445,mh_2019_0000406,853.748,854.896,other,10:46:06,11:00:19:748,11:00:20:896
446,mh_2019_0000406,857.233,857.915,other,10:46:06,11:00:23:233,11:00:23:915
...,...,...,...,...,...,...,...
72519,mh_2019_0000045,10477.308,10491.876,other,22:20:09:000,01:14:46:308,01:14:60:876
72520,mh_2019_0000045,10494.368,10506.790,other,22:20:09:000,01:15:03:368,01:15:15:790
72521,mh_2019_0000045,10507.555,10521.777,other,22:20:09:000,01:15:16:555,01:15:30:777
72522,mh_2019_0000045,10517.061,10528.607,other,22:20:09:000,01:15:26:061,01:15:37:607


Missing columns count:
file_id       0
Start time    0
End time      0
File Start    0
dtype: int64
Rows with 'other' issue:
     File Start    Start time      End time  start_sec   end_sec  duration_sec  \
442   10:46:06  11:00:00:474  11:00:01:080    834.474   835.080         0.606   
443   10:46:06  11:00:01:404  11:00:01:902    835.404   835.902         0.498   
444   10:46:06  11:00:02:941  11:00:03:818    836.941   837.818         0.877   
445   10:46:06  11:00:19:748  11:00:20:896    853.748   854.896         1.148   
446   10:46:06  11:00:23:233  11:00:23:915    857.233   857.915         0.682   
447   10:46:06  11:00:23:352  11:00:23:839    857.352   857.839         0.487   
448   10:46:06  11:00:24:651  11:00:25:376    858.651   859.376         0.725   
449   10:46:06  11:00:24:857  11:00:26:902    858.857   860.902         2.045   
450   10:46:06  11:00:34:402  11:00:34:857    868.402   868.857         0.455   
451   10:46:06  11:00:37:551  11:00:37:898    871.551   871.898 

In [9]:
file_reason = (
    bad.groupby(["file_id", "invalid_reason"])
       .size()
       .reset_index(name="count")
       .sort_values(["count"], ascending=False)
)

# Dominant reason per file
dominant = (
    file_reason.sort_values(["file_id","count"], ascending=[True,False])
              .groupby("file_id")
              .head(1)
              .rename(columns={"invalid_reason":"dominant_reason","count":"dominant_count"})
)

# Total invalid rows per file
total = bad.groupby("file_id").size().reset_index(name="n_invalid_rows")

file_time_report = total.merge(dominant, on="file_id", how="left") \
                        .sort_values("file_id", ascending=False)

display(file_time_report.head(25))
file_time_report.to_csv(OUT/"file_level_time_invalidity_report.csv", index=False)
print("Saved:", OUT/"file_level_time_invalidity_report.csv")


,file_id,n_invalid_rows,dominant_reason,dominant_count
348,mh_2024_0004557,5,other,5
347,mh_2024_0004550,15,other,15
346,mh_2024_0004549,45,other,45
345,mh_2024_0004547,14,other,14
344,mh_2024_0004456,1,other,1
343,mh_2024_0004451,14,other,14
342,mh_2024_0004445,46,other,46
341,mh_2024_0004432,22,other,22
340,mh_2024_0004430,26,other,26
339,mh_2024_0004429,48,other,48


Saved: nmt4k_validation_out/file_level_time_invalidity_report.csv


In [10]:
exceed = bad[bad["invalid_reason"] == "end_exceeds_edf"].copy()
exceed["excess_sec"] = exceed["end_sec"] - exceed["edf_duration_sec"]

print("Excess seconds summary:")
display(exceed["excess_sec"].describe())

# show top examples
cols = ["file_id","File Start","Start time","End time","start_sec","end_sec","edf_duration_sec","excess_sec","label_merged"]
display(exceed[cols].sort_values("excess_sec", ascending=False).head(30))

exceed.to_csv(OUT/"end_exceeds_edf_examples.csv", index=False)
print("Saved:", OUT/"end_exceeds_edf_examples.csv")

exceed[cols].sort_values("excess_sec", ascending=False).to_csv(OUT/"end_exceeds_edf_examples_updated.csv", index=False)
print("Saved:", OUT/"end_exceeds_edf_examples.csv")


KeyError: 'edf_duration_sec'

In [ ]:
to_be_ignored = ['mh_2019_0001271', 'mh_2019_0000316', 'mh_2019_0000583']
